In [1]:
from vla_foundry.models.base_model import BaseModel
model = BaseModel.from_pretrained("TRI-ML/Foundry-Qwen3VLA-2.1B")


/home/jianch2/CS295/vla_foundry/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resizing token embeddings from 151936 to 151937


INFO:root:=> resuming checkpoint 'hf://TRI-ML/Foundry-Qwen3VLA-2.1B/checkpoints/checkpoint_11.pt' (checkpoint 11)


In [2]:
from vla_foundry.models.base_model import BaseModel
model2 = BaseModel.from_pretrained("TRI-ML/Foundry-VLA-1.7B-full")


/home/jianch2/CS295/vla_foundry/.venv/lib/python3.12/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")
INFO:root:=> resuming checkpoint 'hf://TRI-ML/Foundry-VLA-1.7B-full/checkpoints/checkpoint_11.pt' (checkpoint 11)


In [11]:
def count_params(module, trainable_only=False):
    """Count parameters in a PyTorch module, avoiding double-counting shared params."""
    seen = set()
    total = 0

    for p in module.parameters():
        if trainable_only and not p.requires_grad:
            continue

        if id(p) not in seen:
            total += p.numel()
            seen.add(id(p))

    return total


def inspect_diffusion_policy_size(model_or_path, load_if_path=True, verbose=True):
    """
    Inspect total model size, VLM backbone size, and diffusion/action policy size.

    Usage:
        result = inspect_diffusion_policy_size(model)

    or:
        result = inspect_diffusion_policy_size("TRI-ML/Foundry-Qwen3VLA-2.1B")
    """

    # Load model if user passes a Hugging Face path/name
    if isinstance(model_or_path, str):
        if not load_if_path:
            raise ValueError("Received a string path, but load_if_path=False.")

        from vla_foundry.models.base_model import BaseModel
        model = BaseModel.from_pretrained(model_or_path)
    else:
        model = model_or_path

    def fmt(n):
        return f"{n:,} ({n / 1e6:.2f}M)"

    # Total model params
    total_params = count_params(model)
    trainable_total_params = count_params(model, trainable_only=True)

    # VLM backbone params
    vlm_params = None
    trainable_vlm_params = None

    if hasattr(model, "vision_language_backbone"):
        vlm_params = count_params(model.vision_language_backbone)
        trainable_vlm_params = count_params(model.vision_language_backbone, trainable_only=True)

    # Diffusion/action policy modules based on your printed architecture
    diffusion_module_names = [
        "transformer",
        "time_encoding",
        "output_layer",
        "action_encode",
        "condition_encode",
    ]

    diffusion_breakdown = {}
    diffusion_params = 0
    trainable_diffusion_params = 0

    for name in diffusion_module_names:
        if hasattr(model, name):
            module = getattr(model, name)
            n = count_params(module)
            tn = count_params(module, trainable_only=True)

            diffusion_breakdown[name] = {
                "params": n,
                "params_m": n / 1e6,
                "trainable_params": tn,
                "trainable_params_m": tn / 1e6,
            }

            diffusion_params += n
            trainable_diffusion_params += tn

    # Fallback: if VLM exists, non-VLM params should roughly equal policy params
    non_vlm_params = None
    trainable_non_vlm_params = None

    if vlm_params is not None:
        non_vlm_params = total_params - vlm_params
        trainable_non_vlm_params = trainable_total_params - trainable_vlm_params

    result = {
        "total_params": total_params,
        "total_params_m": total_params / 1e6,
        "trainable_total_params": trainable_total_params,
        "trainable_total_params_m": trainable_total_params / 1e6,

        "vlm_params": vlm_params,
        "vlm_params_m": None if vlm_params is None else vlm_params / 1e6,
        "trainable_vlm_params": trainable_vlm_params,
        "trainable_vlm_params_m": None if trainable_vlm_params is None else trainable_vlm_params / 1e6,

        "diffusion_policy_params": diffusion_params,
        "diffusion_policy_params_m": diffusion_params / 1e6,
        "trainable_diffusion_policy_params": trainable_diffusion_params,
        "trainable_diffusion_policy_params_m": trainable_diffusion_params / 1e6,

        "non_vlm_params": non_vlm_params,
        "non_vlm_params_m": None if non_vlm_params is None else non_vlm_params / 1e6,
        "trainable_non_vlm_params": trainable_non_vlm_params,
        "trainable_non_vlm_params_m": None if trainable_non_vlm_params is None else trainable_non_vlm_params / 1e6,

        "diffusion_breakdown": diffusion_breakdown,
    }

    if verbose:
        print("=" * 80)
        print("Model size summary")
        print("=" * 80)
        print(f"Total model params:              {fmt(total_params)}")
        print(f"Trainable total params:          {fmt(trainable_total_params)}")

        if vlm_params is not None:
            print(f"VLM backbone params:             {fmt(vlm_params)}")
            print(f"Trainable VLM backbone params:   {fmt(trainable_vlm_params)}")
            print(f"Non-VLM params:                  {fmt(non_vlm_params)}")
            print(f"Trainable non-VLM params:        {fmt(trainable_non_vlm_params)}")

        print()
        print("Diffusion/action policy")
        print("-" * 80)
        print(f"Diffusion policy params:         {fmt(diffusion_params)}")
        print(f"Trainable diffusion params:      {fmt(trainable_diffusion_params)}")

        print()
        print("Diffusion/action breakdown")
        print("-" * 80)
        for name, stats in diffusion_breakdown.items():
            print(f"{name:25s} {fmt(stats['params'])}")

    return result

In [13]:
stat1 = inspect_diffusion_policy_size(model)

Model size summary
Total model params:              2,543,388,692 (2543.39M)
Trainable total params:          2,543,388,692 (2543.39M)
VLM backbone params:             2,127,534,080 (2127.53M)
Trainable VLM backbone params:   2,127,534,080 (2127.53M)
Non-VLM params:                  415,854,612 (415.85M)
Trainable non-VLM params:        415,854,612 (415.85M)

Diffusion/action policy
--------------------------------------------------------------------------------
Diffusion policy params:         415,854,612 (415.85M)
Trainable diffusion params:      415,854,612 (415.85M)

Diffusion/action breakdown
--------------------------------------------------------------------------------
transformer               411,666,432 (411.67M)
time_encoding             2,048,000 (2.05M)
output_layer              20,500 (0.02M)
action_encode             21,504 (0.02M)
condition_encode          2,098,176 (2.10M)


In [14]:
stat2 = inspect_diffusion_policy_size(model2)

Model size summary
Total model params:              1,852,379,412 (1852.38M)
Trainable total params:          1,852,379,412 (1852.38M)
VLM backbone params:             1,527,374,080 (1527.37M)
Trainable VLM backbone params:   1,527,374,080 (1527.37M)
Non-VLM params:                  325,005,332 (325.01M)
Trainable non-VLM params:        325,005,332 (325.01M)

Diffusion/action policy
--------------------------------------------------------------------------------
Diffusion policy params:         325,005,332 (325.01M)
Trainable diffusion params:      325,005,332 (325.01M)

Diffusion/action breakdown
--------------------------------------------------------------------------------
transformer               308,381,696 (308.38M)
time_encoding             8,192,000 (8.19M)
output_layer              20,500 (0.02M)
action_encode             21,504 (0.02M)
condition_encode          8,389,632 (8.39M)
